# Phase 1 · Notebook 01 — Data Profiling & Data Curation
**Project:** Water Quality Monitoring Analytics — Batch, Distributed & Real-Time DW with OLAP and Data Mining
**Dataset:** Water Quality Portal (WQP) `resultphyschem.csv` — USGS + U.S. EPA + National Water Quality Monitoring Council

This notebook covers **Part A (first half)** of the assignment, *Dataset Understanding and Data Curation*:

| Step | What happens | Assignment reference |
|---|---|---|
| 1 | Configuration, paths, helper functions | — |
| 2 | **Profile the raw file before any cleaning**: missing values in all 81 columns, exact duplicates | §4 Data curation |
| 3 | Load only the analytically useful columns | — |
| 4 | Data-quality checks: duplicates, codes, result values, units, impossible values, timestamps, coordinates, logical duplicates, outliers | §4 |
| 5 | Decide which rows are *analysis-ready* (rows are never silently dropped, each has an explicit exclusion reason) | §4 |
| 6 | Derived attributes: Year, Month, Season, TimeBand, ParameterGroup, ResultBand, ExceedanceFlag, HighRiskFlag, SiteDailyAverage, SampleTypeGroup, RegionGroup | §1 Derived attributes |
| 7 | Reconcile row counts and key measures before vs after curation | §4 |
| 8 | Data-quality report, derived-attribute dictionary | §4 Data-quality report |
| 9 | Save curated dataset → input to `02_exploratory_data_analysis.ipynb` and Phase 2 ETL | §18 Integrated lifecycle |

> **Rule followed throughout:** every cleaning decision is logged with the *attribute*, the *affected-row count*, the *treatment* and the *justification* (no blanket statements such as "all nulls were removed").

## Library guide (simple words)
These are the Python libraries used in this notebook, what each one is for, and the main functions we call from it.

| Library | What it is, in simple words | Main functions used here |
|---|---|---|
| **pandas** (`pd`) | Works with tables (rows and columns), like Excel inside Python. A table is called a **DataFrame**, and one column is a **Series** | `read_csv`, `value_counts`, `groupby`, `to_numeric`, `to_datetime`, `duplicated`, `to_parquet` |
| **numpy** (`np`) | Fast maths on large lists of numbers (arrays). pandas is built on top of it | `np.where`, `np.select`, `np.isnan`, `np.concatenate` |
| **matplotlib** (`plt`) | The basic drawing library. Makes figures, axes, bars, lines | `plt.subplots`, `ax.barh`, `plt.savefig`, `plt.show` |
| **seaborn** (`sns`) | Built on matplotlib. Makes good-looking statistical charts in one line | `sns.set_theme` |
| **pyarrow** | Lets pandas save and read **Parquet** files (a compressed, fast column-based file format, much smaller than CSV) | used by `DataFrame.to_parquet` |
| **pathlib** (`Path`) | Handles folder and file paths in a way that works on Windows, Linux and Kaggle | `Path(...)`, `/` to join paths, `.mkdir`, `.exists` |
| **glob** | Finds files whose names match a pattern, e.g. `*.csv` | `glob.glob(pattern, recursive=True)` |
| **os** | Talks to the operating system | `os.path.getsize` (file size) |
| **time** | Measures how long a step takes | `time.time()` |
| **warnings** | Hides harmless "future version" warning messages so the output stays readable | `warnings.filterwarnings` |

### Key pandas / numpy functions used in this notebook

| Function | What it does |
|---|---|
| `pd.read_csv(file, chunksize=…)` | Reads a CSV. With `chunksize` it reads the file **piece by piece** (e.g. 500,000 rows at a time), so a 2.5 GB file never needs to fit in memory at once |
| `pd.read_csv(file, usecols=…, dtype=…)` | Reads only selected columns and tells pandas the data type of each one, which saves memory |
| `dtype="category"` | Stores repeated text (e.g. "deg C") once and uses small numbers internally. Saves a lot of memory |
| `df.isna()` / `df.notna()` | True/False table showing where values are missing / present |
| `Series.value_counts()` | Counts how many times each value appears |
| `pd.util.hash_pandas_object` | Turns each row into one 64-bit number (a "fingerprint"). Identical rows give identical numbers, which is how exact duplicates are found quickly |
| `Series.duplicated()` | True for every value that already appeared earlier |
| `pd.to_numeric(x, errors="coerce")` | Converts text to numbers. Text that is not a number becomes `NaN` (missing) instead of crashing |
| `pd.to_datetime(x, format=…)` | Converts text like `2024-01-29` to a real date |
| `Series.str.match(regex)` | Checks text against a pattern (e.g. "starts with `<` followed by a digit") |
| `np.where(cond, a, b)` | For every row: if `cond` is true take `a`, else take `b` |
| `np.select([c1, c2, …], [v1, v2, …], default)` | Like a chain of if / elif / else applied to every row at once |
| `df.groupby(cols).agg(...)` | Splits the table into groups (e.g. per parameter) and calculates something for each group (count, mean …) |
| `groupby(...).transform("mean")` | Same as above, but writes the group result back onto **every row** of the group |
| `Series.quantile(q)` | The value below which `q` of the data lies (0.25 = first quartile) |
| `df.to_parquet(path)` | Saves the table as a Parquet file |

## 1. Setup and configuration
The notebook runs both **on Kaggle** and **locally from the GitHub repo**, and finds the dataset in either place.
Set `SAMPLE_NROWS` to a number (e.g. `300_000`) for a quick test run. Keep it `None` for the full analysis.

#### Cell explanation
**What it does:** Imports the libraries, sets the run options, and works out where the data and output folders are.
* `SAMPLE_NROWS = None` means **use the full dataset** (all ~4.96 million rows). A number such as `300_000` would make a quick test run.
* `CHUNK_SIZE = 500_000`: the raw file is read 500k rows at a time to keep memory low.
* `IN_KAGGLE` checks whether the folder `/kaggle/input` exists. If it does, we are on Kaggle and outputs go to `/kaggle/working`.
* `find_raw_csv()` searches for `resultphyschem.csv` (using `glob`) under the Kaggle inputs or the local repo folders.

**Functions:** `Path(...).exists()` checks a path exists. `mkdir(parents=True, exist_ok=True)` creates a folder if it is missing. `os.path.getsize` returns the file size in bytes. `pd.set_option` controls how tables are printed.

**Output to check:** the printed file path and its size (about 2.5 GB).

In [ ]:
import glob
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# ---------------- run configuration ----------------
SAMPLE_NROWS = None        # None = FULL dataset (~4.96M rows). Only set a number (e.g. 300_000) for a quick test
CHUNK_SIZE = 500_000       # rows per chunk while profiling the 2.5 GB raw file
STUDY_START = pd.Timestamp("2024-01-01")   # prescribed period: at least two years
STUDY_END = pd.Timestamp("2025-12-31")

# ---------------- paths (Kaggle or local repo) ----------------
IN_KAGGLE = Path("/kaggle/input").exists()
if IN_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
else:
    # walk up from notebooks/phase1_curation_eda/ to the repo root
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
for d in (PROCESSED_DIR, TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def find_raw_csv():
    # Return the path of resultphyschem.csv wherever it lives.
    candidates = []
    if IN_KAGGLE:
        candidates += sorted(glob.glob("/kaggle/input/**/resultphyschem*.csv", recursive=True))
        if not candidates:  # Kaggle sometimes renames uploads, so fall back to the largest CSV in the inputs
            all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
            candidates += sorted(all_csv, key=os.path.getsize, reverse=True)[:1]
    for p in (PROJECT_ROOT / "data" / "raw" / "resultphyschem.csv",
              PROJECT_ROOT / "resultphyschem" / "resultphyschem.csv"):
        if p.exists():
            candidates.append(str(p))
    if not candidates:
        raise FileNotFoundError("resultphyschem.csv not found. See data/README.md for where to put it.")
    return candidates[0]


RAW_CSV = find_raw_csv()
print("Running on Kaggle :", IN_KAGGLE)
print("Raw file          :", RAW_CSV, f"({os.path.getsize(RAW_CSV) / 1e9:.2f} GB)")
print("Outputs go to     :", PROJECT_ROOT)

#### Cell explanation
**What it does:** Defines the chart style and four small helper functions used throughout the notebook.
* **Fixed colours:** each parameter (temperature, DO, pH, …) always gets the same colour in every chart, so a reader never has to re-learn the legend.
* `save_fig(name)` saves the current chart as a PNG in `reports/figures/`.
* `save_table(df, name)` saves a result table as CSV in `reports/tables/`.
* `log_dq(...)` records **one data-cleaning decision** (problem, column, number of rows, treatment, reason) in the list `DQ_LOG`. That list becomes the Data-Quality Report at the end.
* `map_categories(series, func)` applies a function to each **distinct value** instead of each row. With 5 million rows but only around 40 distinct values, this is thousands of times faster.

**Functions:** `sns.set_theme` sets the global chart look. `plt.savefig` writes a figure to a file. `series.cat.categories` gives the list of distinct values of a category column. `series.cat.codes` gives, for each row, the position of its value in that list.

In [ ]:
# ---------------- plotting style ----------------
# One fixed colour per water-quality parameter, used in every chart of the project
# (colour follows the entity, never its rank).
PARAM_ORDER = ["Temperature, water", "Dissolved oxygen (DO)", "pH", "Turbidity", "Conductivity", "Nitrate"]
PARAM_SHORT = {"Temperature, water": "Temperature", "Dissolved oxygen (DO)": "DO", "pH": "pH",
               "Turbidity": "Turbidity", "Conductivity": "Conductivity", "Nitrate": "Nitrate"}
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
PARAM_COLORS = dict(zip(PARAM_ORDER, PALETTE))
PRIMARY = PALETTE[0]
CRITICAL = "#d03b3b"   # status colour reserved for problems / exceedances

sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#c3c2b7", "grid.color": "#e6e5e0", "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "figure.dpi": 100})


def save_fig(name):
    # Save the current figure to reports/figures so it can be used in the report.
    plt.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")


def save_table(df, name):
    # Save a result table to reports/tables (committed to GitHub as evidence).
    df.to_csv(TABLE_DIR / f"{name}.csv", index=True)


# ---------------- data-quality decision log ----------------
# Every cleaning decision is appended here and becomes the Data-Quality Report (Section 8).
DQ_LOG = []


def log_dq(problem, field, count, treatment, justification):
    DQ_LOG.append({"QualityProblem": problem, "Field": field, "Count": int(count),
                   "Treatment": treatment, "Justification": justification})
    print(f"[DQ] {problem:<38} | {field:<28} | {int(count):>10,} rows | {treatment}")


def map_categories(series, func):
    # Apply `func` once per category instead of once per row (fast on 5M rows).
    # Missing values stay missing.
    cats = pd.Series(series.cat.categories)
    mapped = cats.map(func).to_numpy(dtype=object)
    codes = series.cat.codes.to_numpy()
    out = np.where(codes >= 0, mapped[np.clip(codes, 0, None)], None)
    return pd.Series(out, index=series.index, dtype=object)

## 2. Raw-file profiling (before any cleaning)
The assignment requires profiling the dataset **before** it is cleaned. The 2.5 GB file is read in chunks, with every column as text, so nothing gets converted silently. For each chunk we record:
* the non-null count of **all 81 columns**, giving missing-value counts and percentages
* a 64-bit hash of the *entire row*, used to find **exact duplicates** across the whole file
* a hash of `ResultIdentifier`, used to check that the source primary key is unique

#### Cell explanation
**What it does:** Reads the **entire raw file** once, in chunks, **before any cleaning** (the assignment requires profiling first). For every chunk it:
1. counts non-missing values in all 81 columns (`chunk.notna().sum()`),
2. makes a fingerprint (hash) of every full row, used to find exact duplicates,
3. makes a fingerprint of `ResultIdentifier`, the source's own row ID,
4. counts rows per parameter (`value_counts`), used later for reconciliation.

After the loop, `duplicated()` on the fingerprints marks every row that is a 2nd, 3rd … copy of an earlier row.

**Why `dtype=str`?** All columns are read as text, so pandas cannot silently change anything (e.g. turn `"=6.92"` into missing).

**Functions:** `pd.read_csv(..., chunksize=)` returns an iterator of DataFrames. `pd.util.hash_pandas_object` gives one 64-bit number per row. `np.concatenate` joins the per-chunk arrays.

**Output to check:** total rows (about 4.96M), the number of exact duplicate rows and the number of repeated ResultIdentifiers. On the full file this takes a few minutes.

In [ ]:
t0 = time.time()
n_raw = 0
non_null = None
row_hashes, rid_hashes, char_counts = [], [], []

reader = pd.read_csv(RAW_CSV, dtype=str, chunksize=CHUNK_SIZE, nrows=SAMPLE_NROWS, low_memory=False)
for chunk in reader:
    n_raw += len(chunk)
    nn = chunk.notna().sum()
    non_null = nn if non_null is None else non_null.add(nn, fill_value=0)
    # whole-row hash: two rows with identical values in all 81 columns get the same hash
    row_hashes.append(pd.util.hash_pandas_object(chunk, index=False).to_numpy())
    rid_hashes.append(pd.util.hash_pandas_object(chunk["ResultIdentifier"], index=False).to_numpy())
    char_counts.append(chunk["CharacteristicName"].value_counts())
    RAW_COLUMNS = list(chunk.columns)

row_hashes = np.concatenate(row_hashes)
exact_dup_mask = pd.Series(row_hashes).duplicated(keep="first").to_numpy()   # keep the 1st copy
n_exact_dup = int(exact_dup_mask.sum())
n_dup_rid = int(pd.Series(np.concatenate(rid_hashes)).duplicated().sum())
raw_char_counts = pd.concat(char_counts).groupby(level=0).sum()

print(f"Rows: {n_raw:,} | Columns: {len(RAW_COLUMNS)} | read in {time.time() - t0:.0f}s")
print(f"Exact duplicate rows (all 81 columns identical): {n_exact_dup:,}")
print(f"Repeated ResultIdentifier values               : {n_dup_rid:,}")

#### Cell explanation
**What it does:** Builds the **missing-value table** for all 81 columns: non-null count, missing count and missing %. Important fields are marked, and columns that are ≥ 99.9% empty are listed.

**Functions:** `n_raw - non_null` gives the missing count. `index.isin(list)` returns True for the important columns. `sort_values` orders the table.

**Output to check:** which important fields have many missing values (e.g. start time, media subdivision, collection method). These numbers go straight into the data-quality report.

In [ ]:
# Missing-value profile for ALL columns
missing = pd.DataFrame({"NonNull": non_null.astype(int)})
missing["Missing"] = n_raw - missing["NonNull"]
missing["MissingPct"] = (100 * missing["Missing"] / n_raw).round(2)

# Fields the assignment lists as important (+ a few needed for curation)
IMPORTANT_FIELDS = [
    "MonitoringLocationIdentifier", "ActivityStartDate", "ActivityStartTime/Time",
    "ActivityStartTime/TimeZoneCode", "CharacteristicName", "ResultMeasureValue",
    "ResultMeasure/MeasureUnitCode", "OrganizationIdentifier", "ActivityLocation/LatitudeMeasure",
    "ActivityLocation/LongitudeMeasure", "SampleCollectionMethod/MethodName", "ResultStatusIdentifier",
    "ActivityTypeCode", "ActivityMediaSubdivisionName", "ResultDetectionConditionText",
    "ResultSampleFractionText", "MethodSpeciationName", "ResultIdentifier"]
missing["Important"] = missing.index.isin(IMPORTANT_FIELDS)
save_table(missing, "01_missing_values_all_columns")

empty_cols = missing.index[missing["MissingPct"] >= 99.9].tolist()
print(f"{len(empty_cols)} columns are ≥99.9% empty (not useful for analysis):")
print(", ".join(empty_cols))
missing[missing["Important"]].sort_values("MissingPct", ascending=False)

#### Cell explanation
**What it does:** Draws a horizontal bar chart of the missing % for the important fields, so the gaps can be seen at a glance.

**Functions:** `plt.subplots` creates a figure (`fig`) and drawing area (`ax`). `ax.barh` draws horizontal bars. `ax.text` writes the % label next to each bar.

**Output to check:** long bars are fields we cannot rely on and must treat explicitly (e.g. map to "Not Reported").

In [ ]:
imp = missing[missing["Important"]].sort_values("MissingPct")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp.index, imp["MissingPct"], color=PRIMARY, height=0.6)
for y, v in enumerate(imp["MissingPct"]):
    if v > 0:
        ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=8, color="#52514e")
ax.set_xlabel("% of rows missing")
ax.set_title("Missing values in important fields (raw file)")
ax.set_xlim(0, 105)
ax.grid(axis="y", visible=False)
save_fig("01_missing_important_fields")
plt.show()

**What to read from this:**
* Keys used to identify each result (site, date, characteristic, organisation, result ID) are complete.
* `MonitoringLocationTypeName`, the *site type* listed in the assignment, **is not in the result file**. It lives in the WQP *Station* file and gets added in Phase 2 by joining on `MonitoringLocationIdentifier`. Until then, `LocationType` = "Not Available" and media subdivision is used as a proxy.
* Some columns are nearly empty (e.g. biological/tissue/binary-object fields) because this extract only contains physical/chemical water results. They are left out of the working set.

## 3. Load the working columns
Only the columns needed for curation, EDA and the later dimensional model are loaded. Low-cardinality text gets the `category` dtype, which cuts memory roughly 5–10×. Columns are renamed to short names that match the planned star-schema attributes.

#### Cell explanation
**What it does:** Loads **only the 29 useful columns** (not all 81) into memory, then renames them to short names (e.g. `MonitoringLocationIdentifier` → `SiteId`) that match the planned data-warehouse attributes.
* Repeated text columns get `dtype="category"` (much less memory).
* IDs, dates, values and coordinates are kept as text, because we validate and convert them ourselves in the next steps.
* The exact-duplicate marker from the profiling pass is attached (same row order).

**Functions:** `read_csv(usecols=..., dtype=...)` reads selected columns with set types. `rename(columns=dict)` renames columns. `memory_usage(deep=True)` reports memory in bytes. `assert` stops the notebook if the row count differs from the profiling pass.

**Output to check:** rows loaded = rows profiled, and the memory used.

In [ ]:
COLUMN_MAP = {   # source WQP column            -> short project name
    "ResultIdentifier": "ResultId",
    "OrganizationIdentifier": "OrganizationId",
    "OrganizationFormalName": "OrganizationName",
    "ActivityTypeCode": "ActivityType",
    "ActivityMediaSubdivisionName": "MediaSubdivision",
    "ActivityStartDate": "StartDate",
    "ActivityStartTime/Time": "StartTime",
    "ActivityStartTime/TimeZoneCode": "TimeZone",
    "ActivityEndDate": "EndDate",
    "ActivityEndTime/Time": "EndTime",
    "ActivityDepthHeightMeasure/MeasureValue": "DepthValue",
    "ActivityDepthHeightMeasure/MeasureUnitCode": "DepthUnit",
    "MonitoringLocationIdentifier": "SiteId",
    "MonitoringLocationName": "SiteName",
    "ActivityLocation/LatitudeMeasure": "Latitude",
    "ActivityLocation/LongitudeMeasure": "Longitude",
    "SampleCollectionMethod/MethodName": "CollectionMethod",
    "SampleCollectionEquipmentName": "CollectionEquipment",
    "CharacteristicName": "Characteristic",
    "MethodSpeciationName": "Speciation",
    "ResultSampleFractionText": "SampleFraction",
    "ResultMeasureValue": "ResultRaw",
    "ResultMeasure/MeasureUnitCode": "UnitRaw",
    "ResultDetectionConditionText": "DetectionCondition",
    "MeasureQualifierCode": "QualifierCode",
    "ResultStatusIdentifier": "ResultStatus",
    "ResultValueTypeName": "ValueType",
    "HydrologicEvent": "HydrologicEvent",
    "ProviderName": "Provider",
}
# These stay as plain text (high cardinality, or parsed/validated explicitly later)
TEXT_COLS = ["ResultIdentifier", "ActivityStartDate", "ActivityEndDate", "ResultMeasureValue",
             "ActivityDepthHeightMeasure/MeasureValue",
             "ActivityLocation/LatitudeMeasure", "ActivityLocation/LongitudeMeasure"]
dtypes = {c: (str if c in TEXT_COLS else "category") for c in COLUMN_MAP}

t0 = time.time()
df = pd.read_csv(RAW_CSV, usecols=list(COLUMN_MAP), dtype=dtypes, nrows=SAMPLE_NROWS, low_memory=False)
df = df.rename(columns=COLUMN_MAP)[list(COLUMN_MAP.values())]
assert len(df) == n_raw, "row count differs from the profiling pass"
df["IsExactDuplicate"] = exact_dup_mask          # same row order as the profiling pass

print(f"Loaded {len(df):,} rows x {df.shape[1]} cols in {time.time() - t0:.0f}s, "
      f"memory = {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head(3)

## 4. Data-quality assessment and cleaning decisions
### 4.1 Exact duplicates and the source key
An **exact duplicate** has the same value in all 81 columns, so it is the same result submitted twice and is removed (first copy kept).
A **repeated `ResultIdentifier`** with *different* content gets looked into, not deleted.

#### Cell explanation
**What it does:** Removes **exact duplicate** rows (identical in every column) and keeps the first copy. Then it counts `ResultIdentifier` values that are repeated with different content, which are investigated but **kept**.

**Functions:** `df.loc[~mask]` keeps rows where the mask is False (`~` means NOT). `reset_index(drop=True)` renumbers the rows 0, 1, 2 …. `log_dq(...)` records the decision.

**Output to check:** the `[DQ]` lines printed. These are entries in the data-quality report.

In [ ]:
N_RAW = len(df)
log_dq("Duplicate record (exact)", "all 81 columns", n_exact_dup, "Remove (keep first copy)",
       "Identical in every source column, so it is the same result loaded twice; keeping it would double-count.")

df = df.loc[~df["IsExactDuplicate"]].drop(columns="IsExactDuplicate").reset_index(drop=True)
rid_rep = int(df["ResultId"].duplicated().sum())
log_dq("Repeated source key", "ResultIdentifier", rid_rep, "Investigate / retain",
       "Same ResultIdentifier but different content. Keep both rows, because the warehouse uses its own surrogate key.")
print(f"Rows after exact-duplicate removal: {len(df):,}")

### 4.2 Categorical / code validation
Codes are checked against the WQP domain values (https://www.waterqualitydata.us/public_srsnames/).
* **ActivityType** → `SampleTypeGroup` (derived). *Quality-control* activities (blanks, replicates, calibration checks) measure the procedure rather than the river, so they are kept in the table but **excluded from analysis**.
* **ResultStatus** `Rejected` is excluded. Preliminary / Provisional / Raw / Unreviewed results are kept but flagged.
* **QualifierCode** containing `SUS` (*suspect*, as flagged by the data provider) is excluded from analysis.
* Missing **MediaSubdivision** / **CollectionMethod** map to an explicit `Not Reported` member rather than being dropped.

#### Cell explanation
**What it does:** Checks that the dataset contains only the 6 expected parameters, then prints the **distinct codes and their counts** for the main categorical columns (activity type, result status, media, value type, provider, detection condition).

**Functions:** `set(...)` holds unique values, and `a - b` gives the values in `a` but not in `b`. `value_counts(dropna=False)` counts each code, including missing (NaN). `nunique()` gives the number of distinct values.

**Output to check:** unexpected or strange codes (e.g. "Unknown", "Rejected", quality-control activity types) that need a decision in the next cell.

In [ ]:
EXPECTED_PARAMS = set(PARAM_ORDER)
found = set(df["Characteristic"].dropna().unique())
print("Characteristics found:", sorted(found))
print("Unexpected characteristics:", sorted(found - EXPECTED_PARAMS) or "none")

for col in ["ActivityType", "ResultStatus", "MediaSubdivision", "ValueType", "Provider", "DetectionCondition"]:
    vc = df[col].value_counts(dropna=False)
    print(f"\n--- {col}: {df[col].nunique()} codes ---")
    print(vc.head(12).to_string())

#### Cell explanation
**What it does:** Makes the categorical-code decisions:
* **`SampleTypeGroup`** (derived): groups the 36 activity types into 5 groups: Quality Control, Logger / Continuous, Field Measurement, Lab Sample, Unknown.
* **Quality-control** rows (blanks, replicates, calibration checks) test the measuring process, not the river, so they are **marked for exclusion** from analysis.
* **Rejected** results are marked for exclusion. Preliminary/Raw/Unreviewed results are kept with the flag `IsProvisional`.
* **Suspect** values (qualifier code `SUS`, flagged by the data provider) are marked for exclusion.
* Missing descriptive fields (media, method, equipment, fraction, time zone) are filled with an explicit **"Not Reported"** member instead of dropping rows.

**Functions:** `map_categories` (our helper) applies the grouping function once per code. `.eq("x")` is the same as `== "x"`. `.isin(list)` checks membership. `cat.add_categories` adds a new allowed value to a category column, and `fillna` fills the missing values with it.

In [ ]:
def sample_type_group(activity_type):
    # Derived attribute SampleTypeGroup, grouped from the WQP ActivityTypeCode domain.
    a = str(activity_type)
    if "Quality Control" in a:
        return "Quality Control"
    if "Data Logger" in a or "Continuous" in a:
        return "Logger / Continuous"
    if a.startswith("Field Msr/Obs"):
        return "Field Measurement"
    if a.startswith("Sample"):
        return "Lab Sample"
    return "Unknown"


df["SampleTypeGroup"] = map_categories(df["ActivityType"], sample_type_group).astype("category")
print(df["SampleTypeGroup"].value_counts().to_string())

is_qc = df["SampleTypeGroup"].eq("Quality Control").to_numpy()
log_dq("Non-environmental record (QC)", "ActivityTypeCode", is_qc.sum(), "Retain, exclude from analysis",
       "Blanks, replicates and calibration checks describe the measurement process, not water quality.")
log_dq("Unknown dimension/code", "ActivityTypeCode = 'Unknown'", df["SampleTypeGroup"].eq("Unknown").sum(),
       "Map to 'Unknown' member", "Activity type not reported. The result itself is valid, so it is kept.")

is_rejected = df["ResultStatus"].eq("Rejected").to_numpy()
log_dq("Invalid status code", "ResultStatusIdentifier = Rejected", is_rejected.sum(), "Exclude from analysis",
       "Data provider rejected the result.")
PROVISIONAL = ["Preliminary", "Provisional", "Raw", "Unreviewed"]
df["IsProvisional"] = df["ResultStatus"].isin(PROVISIONAL)
log_dq("Not-yet-final status", "ResultStatusIdentifier", df["IsProvisional"].sum(), "Retain + flag IsProvisional",
       "Not rejected, and most recent data is still under review. Flag so it can be filtered in BI.")

is_suspect = map_categories(df["QualifierCode"], lambda q: "SUS" in str(q).split(";")).fillna(False).astype(bool).to_numpy()
log_dq("Provider-flagged suspect value", "MeasureQualifierCode contains SUS", is_suspect.sum(),
       "Retain, exclude from analysis", "Data provider marked the value as suspect.")

for col in ["MediaSubdivision", "CollectionMethod", "CollectionEquipment", "SampleFraction", "TimeZone"]:
    n_na = int(df[col].isna().sum())
    df[col] = df[col].cat.add_categories("Not Reported").fillna("Not Reported")
    log_dq("Missing important field", col, n_na, "Map to 'Not Reported' member",
           "Descriptive attribute. The result is still valid without it, and dropping would lose facts.")

### 4.3 Result value parsing (`ResultMeasureValue`)
The raw value is text. Each row ends up in one of these groups:

| Parse status | Example | Treatment |
|---|---|---|
| Numeric | `8.4` | used |
| Numeric after stripping `=` | `=6.92` | used (the `=` is a data-entry artefact) |
| Censored | `<1`, `>230` | not usable as an exact value. Kept with the limit in `CensoredLimit`, excluded from numeric statistics |
| Text placeholder | `Null`, `NF`, `ND`, `Not Reported`, `clear` | excluded from numeric analysis |
| Missing | *(blank)* | excluded from numeric analysis. Rows with a `DetectionCondition` count as *non-detects* |

#### Cell explanation
**What it does:** Converts the measured value (`ResultMeasureValue`, stored as text) into a number, and records **why** a value could not be converted:
* `8.4` → number
* `=6.92` → strip the `=`, then number
* `<1`, `>230` → **censored** (only a limit is known). The limit is stored in `CensoredLimit`, and the row is not used as an exact value
* `Null`, `NF`, `Not Reported` … → text placeholder (no value)
* blank → missing

**Functions:** `pd.to_numeric(errors="coerce")` turns non-numbers into NaN. `.str.strip()` removes spaces. `.str.match(r"^[<>]\s*\d")` is a regular expression meaning "starts with < or >, then a digit". `.str.lstrip("=")` removes a leading `=`.

**Output to check:** how many values fall into each `ParseStatus`, and which text placeholders are most common.

In [ ]:
raw = df["ResultRaw"]
value = pd.to_numeric(raw, errors="coerce")
non_numeric = raw.notna() & value.isna()
txt = raw[non_numeric].str.strip()

eq_prefix = txt.str.match(r"^=\s*-?\d")                       # '=6.92'
value.loc[txt.index[eq_prefix]] = pd.to_numeric(txt[eq_prefix].str.lstrip("=").str.strip(), errors="coerce")
censored = txt.str.match(r"^[<>]\s*=?\s*\d")                  # '<1', '> 230'

parse_status = pd.Series("Numeric", index=df.index, dtype=object)
parse_status[raw.isna()] = "Missing"
parse_status[txt.index[eq_prefix]] = "Numeric ('=' stripped)"
parse_status[txt.index[censored]] = "Censored (<, >)"
parse_status[txt.index[~eq_prefix & ~censored]] = "Text placeholder"
df["ParseStatus"] = parse_status.astype("category")
df["CensoredLimit"] = np.nan
df.loc[txt.index[censored], "CensoredLimit"] = pd.to_numeric(
    txt[censored].str.replace(r"[<>=\s]", "", regex=True), errors="coerce")
df["ValueNumeric"] = value

print(df["ParseStatus"].value_counts().to_string())
print("\nMost common text placeholders:")
print(txt[~eq_prefix & ~censored].value_counts().head(15).to_string())

n_missing_val = int(raw.isna().sum())
n_nondetect = int((raw.isna() & df["DetectionCondition"].notna()).sum())
log_dq("Missing important field", "ResultMeasureValue", n_missing_val, "Retain, exclude from numeric analysis",
       f"No value to analyse. {n_nondetect:,} of these carry a DetectionCondition (non-detect), which is kept for coverage counts.")
log_dq("Invalid numeric value", "ResultMeasureValue '=x' prefix", eq_prefix.sum(), "Correct (strip '=')",
       "The number is valid. The '=' is a formatting artefact.")
log_dq("Invalid numeric value", "ResultMeasureValue censored", censored.sum(), "Flag, exclude from numeric stats",
       "'<x' gives only a bound, not a measurement. Limit kept in CensoredLimit.")
log_dq("Invalid numeric value", "ResultMeasureValue text", (~eq_prefix & ~censored).sum(),
       "Exclude from numeric analysis", "Placeholders such as 'Null', 'NF', 'Not Reported' carry no measurement.")

### 4.4 Unit standardisation
The same parameter is reported in many units (°F vs °C, µS/cm vs mS/cm, nitrate *as N* vs *as NO₃*). Every parameter is converted to **one canonical unit** with documented factors:

| Parameter | Canonical unit | Conversions |
|---|---|---|
| Temperature | °C | °F → (°F − 32) × 5/9 |
| Dissolved oxygen | mg/L | ppm = mg/L. **% saturation is a different measure**, so it is excluded and not converted |
| pH | pH units | blank unit / `std units` / `nu` are all pH units. `mV`, `Mole/L` etc. are invalid for pH |
| Turbidity | NTU | NTU ≈ FNU ≈ FTU ≈ NTRU ≈ JTU (treated as approximately equivalent nephelometric/turbidimetric units, a documented approximation). `cm` (Secchi tube) and `mg/L` are not turbidity |
| Conductivity | µS/cm | µmho/cm = µS/cm, mS/cm × 1000, mho/cm × 10⁶, µS/m ÷ 100 |
| Nitrate | mg/L as N | as NO₃ × 0.2259 (14.007/62.004), µg/L ÷ 1000, µmol/L or µeq/L × 0.014007. mg/L with **no speciation is assumed to be as N** (dominant reporting convention, see the speciation counts) |

Unit/parameter pairs not covered above (e.g. *pH in mV*, *temperature in cm*) are **unconvertible** and excluded from numeric analysis, but stay in the table.

#### Cell explanation
**What it does:** **Unit standardisation.** The same parameter is reported in different units (°F and °C, µS/cm and mS/cm, nitrate "as NO3" and "as N"), and they cannot be compared until they are converted. Each parameter gets one canonical unit:
`ValueStd = value × multiplier + offset`, e.g. °F → °C uses multiplier 5/9 and offset −32×5/9.
* DO in **%** (saturation) is a **different measure**, not convertible to mg/L, so it is marked "Different measure".
* Unit/parameter pairs that make no sense (pH in mV, temperature in cm) are marked "Unconvertible / invalid unit".
* The table at the end lists every (parameter, unit) pair, its row count and its status. It is saved as evidence.

**Functions:** `np.full(n, np.nan)` creates an array filled with NaN. Boolean masks such as `(char == p) & (unit == u)` select rows (`&` = AND, `|` = OR). `np.select` assigns the status labels. `groupby([...]).size()` counts rows per pair.

In [ ]:
# (multiplier, offset): standard_value = value * multiplier + offset.   None key = unit missing.
UNIT_CONVERSIONS = {
    "Temperature, water": {"deg C": (1, 0), "deg F": (5 / 9, -32 * 5 / 9)},
    "Dissolved oxygen (DO)": {"mg/L": (1, 0), "ppm": (1, 0), "mg/l": (1, 0)},
    "pH": {None: (1, 0), "std units": (1, 0), "std*^*^*^*^*": (1, 0), "nu": (1, 0), "None": (1, 0)},
    "Turbidity": {"NTU": (1, 0), "FNU": (1, 0), "FTU": (1, 0), "NTRU": (1, 0), "JTU": (1, 0)},
    "Conductivity": {"uS/cm": (1, 0), "umho/cm": (1, 0), "mS/cm": (1000, 0), "mho/cm": (1e6, 0), "uS/m": (0.01, 0)},
    "Nitrate": {"mg/L": (1, 0), "ppm": (1, 0), "mg/l as N": (1, 0), "mg/l asNO3": (0.2259, 0),
                "ug/L": (0.001, 0), "umol/L": (0.014007, 0), "ueq/L": (0.014007, 0)},
}
CANONICAL_UNIT = {"Temperature, water": "deg C", "Dissolved oxygen (DO)": "mg/L", "pH": "pH units",
                  "Turbidity": "NTU", "Conductivity": "uS/cm", "Nitrate": "mg/L as N"}
DIFFERENT_MEASURE = {("Dissolved oxygen (DO)", "%")}      # same characteristic, different quantity
NO3_TO_N = 0.2259

mult = np.full(len(df), np.nan)
offs = np.zeros(len(df))
char = df["Characteristic"]
unit = df["UnitRaw"]
for param, rules in UNIT_CONVERSIONS.items():
    is_param = (char == param).to_numpy()
    for u, (m, o) in rules.items():
        sel = is_param & (unit.isna().to_numpy() if u is None else (unit == u).to_numpy())
        mult[sel], offs[sel] = m, o

# Nitrate speciation: mass units reported "as NO3" are converted to "as N";
# NO2 / NH3 / NO2+NO3 speciations are a different compound, so they are unconvertible.
is_nitrate = (char == "Nitrate").to_numpy()
mass_unit = unit.isin(["mg/L", "ppm", "ug/L"]).to_numpy()
spec = df["Speciation"]
mult[is_nitrate & mass_unit & (spec == "as NO3").to_numpy()] *= NO3_TO_N
mult[is_nitrate & spec.isin(["as NO2", "as NH3", "as NO2+NO3"]).to_numpy()] = np.nan

different = np.zeros(len(df), dtype=bool)
for p, u in DIFFERENT_MEASURE:
    different |= ((char == p) & (unit == u)).to_numpy()

df["ValueStd"] = df["ValueNumeric"].to_numpy() * mult + offs
df["UnitStd"] = char.map(CANONICAL_UNIT).astype("category")
unit_status = np.select(
    [different, np.isnan(mult), (mult == 1) & (offs == 0)],
    ["Different measure", "Unconvertible / invalid unit", "Standard"], default="Converted")
df["UnitStatus"] = pd.Categorical(unit_status)

unit_table = (df.groupby(["Characteristic", "UnitRaw", "UnitStatus"], observed=True, dropna=False)
                .size().rename("Rows").reset_index().sort_values(["Characteristic", "Rows"], ascending=[True, False]))
save_table(unit_table.set_index("Characteristic"), "01_unit_standardisation")
print(df["UnitStatus"].value_counts().to_string())
unit_table.head(40)

#### Cell explanation
**What it does:** Writes the unit decisions into the data-quality log: how many rows were converted, how many are "different measure" (DO %) and how many have an invalid unit.

**Functions:** `(mask1 & mask2).sum()` counts rows where both conditions are true (True counts as 1).

In [ ]:
has_value = df["ValueNumeric"].notna().to_numpy()
log_dq("Inconsistent unit", "ResultMeasure/MeasureUnitCode", (has_value & (unit_status == "Converted")).sum(),
       "Correct (convert to canonical unit)", "°F→°C, mS→µS, NO3→N etc. so values are comparable.")
log_dq("Different measure, same name", "DO reported in %", (has_value & different).sum(),
       "Retain, exclude from mg/L analysis", "% saturation is not a concentration, so it cannot be converted to mg/L without temperature/pressure.")
log_dq("Unknown dimension/code", "Unit invalid for parameter", (has_value & (unit_status == "Unconvertible / invalid unit")).sum(),
       "Retain, exclude from numeric analysis", "Unit missing or incompatible (e.g. pH in mV), so the value cannot be interpreted.")

### 4.5 Zero, negative and physically impossible values
Zero and negative values are **inspected first** (a zero DO or zero turbidity can be real). Then values outside a *physically possible* range (in the canonical unit) are flagged as **impossible**:

| Parameter | Possible range | Reason |
|---|---|---|
| Temperature | −5 to 50 °C | liquid water in natural/effluent settings, supercooled brine down to about −2 °C |
| DO | 0 to 30 mg/L | supersaturation rarely exceeds about 20 mg/L |
| pH | 0 to 14 | pH scale |
| Turbidity | 0 to 10,000 NTU | negative is impossible. Extreme storm values can reach thousands |
| Conductivity | 0 to 100,000 µS/cm | seawater about 55,000, brines higher |
| Nitrate | 0 to 100 mg/L N | far above any reported surface-water value |

#### Cell explanation
**What it does:** Looks at **zero and negative** values first (zero DO or zero turbidity can be real), then flags values outside a **physically possible range** as impossible, e.g. pH must be 0–14, and turbidity cannot be negative. Impossible values come from typing or unit errors.

**Functions:** `np.flatnonzero(mask)` gives the row positions where the mask is True. `.min()` / `.max()` find the extremes. `pd.DataFrame(rows)` builds the summary table from a list of dictionaries.

**Output to check:** the table of zeros, negatives, below-min and above-max per parameter, plus the real min/max. This shows **how bad** the raw extremes were.

In [ ]:
PLAUSIBLE_RANGE = {"Temperature, water": (-5, 50), "Dissolved oxygen (DO)": (0, 30), "pH": (0, 14),
                   "Turbidity": (0, 10_000), "Conductivity": (0, 100_000), "Nitrate": (0, 100)}

usable = df["ValueStd"].notna() & (df["UnitStatus"].isin(["Standard", "Converted"]))
rows = []
impossible = np.zeros(len(df), dtype=bool)
for p, (lo, hi) in PLAUSIBLE_RANGE.items():
    m = (usable & (df["Characteristic"] == p)).to_numpy()
    v = df["ValueStd"].to_numpy()[m]
    bad = (v < lo) | (v > hi)
    impossible[np.flatnonzero(m)[bad]] = True
    rows.append({"Parameter": p, "Values": m.sum(), "Zero": (v == 0).sum(), "Negative": (v < 0).sum(),
                 "BelowMin": (v < lo).sum(), "AboveMax": (v > hi).sum(), "Min": v.min() if len(v) else np.nan,
                 "Max": v.max() if len(v) else np.nan, "AllowedRange": f"{lo} to {hi}"})
df["IsImpossible"] = impossible
range_table = pd.DataFrame(rows).set_index("Parameter")
save_table(range_table, "01_zero_negative_impossible")
log_dq("Invalid numeric value", "ValueStd outside physical range", impossible.sum(), "Retain, exclude from analysis",
       "Physically impossible values (e.g. pH 25, turbidity < 0) are data-entry or unit errors.")
range_table

### 4.6 Timestamps
Checks:
1. `ActivityStartDate` parses and falls inside the prescribed study period.
2. `ActivityStartTime` has a valid `HH:MM:SS` format.
3. **Midnight placeholders**: `00:00:00` on a *discrete* field visit is usually a placeholder for "time not recorded". For logger/continuous data midnight is a real reading.
4. A time zone is present when a time is present. A UTC timestamp (`StartDateTimeUTC`) is derived for chronological replay in the Phase 2 real-time simulation.
5. End date/time not earlier than start (event ordering). Durations are **not** derived in Phase 1, because more than half of the rows have no end time.

Hour-of-day stays in **local time**, because diurnal patterns (e.g. DO peaking in the afternoon) follow local sun time.

#### Cell explanation
**What it does:** Validates dates and times:
1. Converts `StartDate` text into real dates, and counts invalid dates and dates outside 2024–2025.
2. Checks that the time has the form `HH:MM:SS` and takes the **hour**.
3. **Midnight placeholder:** `00:00:00` on a one-off field visit usually means "time not written down", so it is treated as unknown. Data loggers really do record at midnight, so they keep it.
4. Builds a local timestamp and a **UTC timestamp** (local time minus the time-zone offset, e.g. EST = −5 h). UTC gives a single, correct ordering for the Phase 2 real-time replay.
5. Counts records whose end time is earlier than the start time (broken event order).

**Functions:** `pd.to_datetime(format="%Y-%m-%d", errors="coerce")` converts dates, with bad ones becoming NaT (missing date). `pd.to_timedelta(seconds, unit="s")` turns seconds into a time span, which can be added to a date. `.astype("Int8")` is a small integer type that still allows missing values.

In [ ]:
df["StartDate"] = pd.to_datetime(df["StartDate"], format="%Y-%m-%d", errors="coerce")
df["EndDate"] = pd.to_datetime(df["EndDate"], format="%Y-%m-%d", errors="coerce")
bad_date = df["StartDate"].isna().to_numpy()
out_period = ((df["StartDate"] < STUDY_START) | (df["StartDate"] > STUDY_END)).to_numpy()
print("Date range:", df["StartDate"].min().date(), "to", df["StartDate"].max().date())

# time format and hour (computed per category, fast)
valid_time = map_categories(df["StartTime"], lambda t: bool(pd.notna(t) and len(str(t)) == 8
                                                            and str(t)[2] == ":" and str(t)[:2].isdigit()
                                                            and int(str(t)[:2]) < 24)).fillna(False).astype(bool)
hour = pd.to_numeric(map_categories(df["StartTime"], lambda t: str(t)[:2]), errors="coerce")


def to_seconds(t):
    # 'HH:MM:SS' -> seconds after midnight (None if the text is not a valid time)
    try:
        h, m, sec = (int(x) for x in str(t).split(":"))
        return h * 3600 + m * 60 + sec
    except ValueError:
        return None


start_sec = pd.to_numeric(map_categories(df["StartTime"], to_seconds), errors="coerce")
end_sec = pd.to_numeric(map_categories(df["EndTime"], to_seconds), errors="coerce")
is_midnight = df["StartTime"].eq("00:00:00").to_numpy()
discrete = ~df["SampleTypeGroup"].eq("Logger / Continuous").to_numpy()
midnight_placeholder = is_midnight & discrete

df["TimeKnown"] = valid_time.to_numpy() & ~midnight_placeholder
df["Hour"] = hour.where(df["TimeKnown"]).astype("Int8")

# UTC timestamp (for the streaming replay order in Phase 2)
TZ_OFFSET_H = {"EST": -5, "EDT": -4, "CST": -6, "CDT": -5, "MST": -7, "MDT": -6, "PST": -8, "PDT": -7,
               "AKST": -9, "AKDT": -8, "HST": -10, "HAST": -10, "AST": -4, "UTC": 0, "GMT": 0}
offset = pd.to_numeric(map_categories(df["TimeZone"], TZ_OFFSET_H.get), errors="coerce")
local_dt = df["StartDate"] + pd.to_timedelta(start_sec.where(df["TimeKnown"]), unit="s")
df["StartDateTimeLocal"] = local_dt
df["StartDateTimeUTC"] = local_dt - pd.to_timedelta(offset, unit="h")

no_tz = (df["TimeKnown"] & df["TimeZone"].eq("Not Reported")).to_numpy()
end_dt = df["EndDate"] + pd.to_timedelta(end_sec.fillna(0), unit="s")
end_before_start = (df["EndDate"].notna() & (end_dt < local_dt.fillna(df["StartDate"]))).to_numpy()

log_dq("Invalid timestamp", "ActivityStartDate unparseable", bad_date.sum(), "Exclude", "No date, so the fact cannot be placed in DimDate.")
log_dq("Invalid timestamp", "ActivityStartDate outside study period", (out_period & ~bad_date).sum(), "Exclude",
       f"Outside the prescribed {STUDY_START.date()} to {STUDY_END.date()} window.")
log_dq("Missing important field", "ActivityStartTime/Time", df["StartTime"].isna().sum(), "Retain, TimeBand='Unknown'",
       "Date is known, so the result is still usable for daily/monthly analysis.")
log_dq("Invalid timestamp", "Midnight placeholder time (discrete visit)", midnight_placeholder.sum(),
       "Correct (treat time as unknown)", "00:00:00 on a discrete field visit almost always means 'not recorded'. Keeping it would create a false midnight peak.")
log_dq("Missing important field", "TimeZone while time present", no_tz.sum(), "Retain, UTC left empty",
       "Local hour is still valid. Only the UTC conversion is impossible.")
log_dq("Invalid timestamp or duration", "End date/time earlier than start", end_before_start.sum(),
       "Retain, no duration derived", "Event ordering is broken, so durations are not computed in Phase 1.")

### 4.7 Coordinates and the derived `RegionGroup`
Latitude/longitude get checked for being missing, non-numeric, out of range, at (0, 0), or carrying a **likely sign error** (positive longitude at mainland-US latitudes).

The result file has **no state field**, so `RegionGroup` is derived from coordinates using **approximate longitudinal bands** (documented, *not* official boundaries):

| RegionGroup | Rule |
|---|---|
| Alaska | lat ≥ 51 and lon ≤ −129 |
| Hawaii / Pacific Islands | lon ≤ −150 and lat < 30, or lon > 130 (Guam / N. Mariana Is.) |
| Caribbean | 17 ≤ lat ≤ 19.5 and −68 ≤ lon ≤ −64 (Puerto Rico / USVI) |
| West (Pacific) | lon ≤ −114 |
| Mountain / Plains | −114 < lon ≤ −100 |
| Central | −100 < lon ≤ −87.5 |
| East | lon > −87.5 |
| Unknown | missing or invalid coordinates |

*Phase 2 replaces this with the authoritative state code from the WQP Station service.*

#### Cell explanation
**What it does:** Cleans latitude/longitude and derives **`RegionGroup`**.
* Coordinates that are missing, non-numeric, out of range, exactly 0, or have a **positive longitude at US latitudes** (probably a missing minus sign) are set to missing. They are **not** "fixed", because we cannot be sure of the true value.
* The file has no state column, so `RegionGroup` is built from longitude bands (West, Mountain/Plains, Central, East) plus Alaska, Hawaii/Pacific and Caribbean. This is an **approximation**, documented as such, and replaced with the real state in Phase 2.
* It also counts sites that report more than one latitude.

**Functions:** `Series.between(a, b)` returns True when a ≤ value ≤ b. `.abs()` gives the absolute value. `np.select` applies the region rules in order (first match wins). `groupby("SiteId")["Latitude"].nunique()` counts distinct latitudes per site.

In [ ]:
lat = pd.to_numeric(df["Latitude"], errors="coerce")
lon = pd.to_numeric(df["Longitude"], errors="coerce")
coord_missing = df["Latitude"].isna() | df["Longitude"].isna()
coord_nonnum = ~coord_missing & (lat.isna() | lon.isna())
coord_range = (lat.abs() > 90) | (lon.abs() > 180)
coord_zero = (lat == 0) | (lon == 0)
pacific_islands = (lon > 130) & (lat.between(10, 25))
sign_error = (lon > 0) & ~pacific_islands & lat.between(24, 72)
coord_invalid = coord_nonnum | coord_range | coord_zero | sign_error | coord_missing

df["Latitude"], df["Longitude"] = lat.where(~coord_invalid), lon.where(~coord_invalid)
df["CoordinateValid"] = ~coord_invalid

la, lo = df["Latitude"], df["Longitude"]
df["RegionGroup"] = pd.Categorical(np.select(
    [la.isna(),
     (la >= 51) & (lo <= -129),
     ((lo <= -150) & (la < 30)) | (lo > 130),
     la.between(17, 19.5) & lo.between(-68, -64),
     lo <= -114, lo <= -100, lo <= -87.5],
    ["Unknown", "Alaska", "Hawaii / Pacific Islands", "Caribbean", "West (Pacific)", "Mountain / Plains", "Central"],
    default="East"))

log_dq("Incomplete geographic identifier", "Latitude/Longitude missing", coord_missing.sum(),
       "Retain, RegionGroup='Unknown'", "The result is valid. Only the map and region analysis lose it.")
log_dq("Incomplete geographic identifier", "Lat/Long zero / out of range / non-numeric",
       (coord_nonnum | coord_range | coord_zero).sum(), "Set to missing, RegionGroup='Unknown'", "Impossible coordinates.")
log_dq("Incomplete geographic identifier", "Positive longitude at US latitudes", sign_error.sum(),
       "Set to missing (not auto-corrected)", "Probable missing minus sign. Not corrected silently because the true sign cannot be confirmed.")

multi_coord = df.groupby("SiteId", observed=True)["Latitude"].nunique()
print(f"Sites with more than one latitude value: {(multi_coord > 1).sum():,} of {len(multi_coord):,}"
      " (activity-level coordinates can differ slightly per visit; site coordinate = median in the EDA)")
print(df["RegionGroup"].value_counts().to_string())

### 4.8 Logical duplicates
A **logical duplicate** is a different row (different `ResultId`) that describes *the same observation*: same site, date, **time**, characteristic, fraction, depth, activity type **and the same standardised value**. This usually comes from the same reading being submitted twice (e.g. once by a lab, once by a program office). The logic only applies to rows **with a known time**, because without a time two identical values on the same day can be genuine separate readings.

Same key but *different values* = **conflicting** records. They get reported, not removed: they are typically replicate readings or depth-profile points without a depth value.

#### Cell explanation
**What it does:** Finds **logical duplicates**: different rows (different ID) that describe **the same observation**, meaning the same site, date, time, parameter, fraction, depth, activity type **and the same value**. This happens when one reading is submitted twice by two programmes. Only rows with a known time are checked, because without a time two equal values on one day can be genuine separate readings.
It also counts **conflicting** records: same key, but **different** values (likely replicate readings). These are reported but kept.

**Functions:** `DataFrame.duplicated(subset, keep="first")` marks every repeat after the first. `keep=False` marks **all** members of a repeated group. `del sub` frees the temporary table's memory.

In [ ]:
KEY = ["SiteId", "StartDate", "StartTime", "Characteristic", "SampleFraction", "DepthValue", "ActivityType"]
timed = df["TimeKnown"].to_numpy() & df["ValueStd"].notna().to_numpy()
sub = df.loc[timed, KEY + ["ValueStd"]]
logical_dup = np.zeros(len(df), dtype=bool)
logical_dup[np.flatnonzero(timed)] = sub.duplicated(KEY + ["ValueStd"], keep="first").to_numpy()
conflicting = np.zeros(len(df), dtype=bool)
conflicting[np.flatnonzero(timed)] = (sub.duplicated(KEY, keep=False) & ~sub.duplicated(KEY + ["ValueStd"], keep=False)).to_numpy()
df["IsLogicalDuplicate"] = logical_dup
df["HasConflictingValue"] = conflicting
del sub

log_dq("Duplicate or repeated record (logical)", "site+datetime+parameter+depth+value", logical_dup.sum(),
       "Retain, exclude from analysis", "The same observation reported more than once would bias averages and counts.")
log_dq("Conflicting repeated record", "same key, different value", conflicting.sum(), "Investigate / retain",
       "Likely replicate readings or depth profiles without depth. Both values are plausible, so neither is preferred.")

## 5. Analysis-ready flag and exclusion reason
No row is deleted (apart from exact duplicates). Instead, each row gets **one** `ExclusionReason` (the first matching rule, in the order below), and `IsAnalysisReady = True` when the reason is *Included*. The warehouse can still load every row, with BI and ML filtering on the flag.

#### Cell explanation
**What it does:** Gives every row exactly **one** `ExclusionReason`: the first rule (in the order listed) that applies, or **"Included"** if none do. `IsAnalysisReady = True` only for "Included" rows.
Nothing is deleted here. The warehouse in Phase 2 can still load every row, while EDA and ML use only analysis-ready rows. This avoids the blanket "all nulls were removed" approach the assignment forbids.

**Functions:** `np.select(conditions, labels, default="Included")` picks the first true condition for each row. `pd.Categorical(..., categories=[...])` fixes the label order for the tables.

**Output to check:** the share of analysis-ready rows and the biggest exclusion reasons.

In [ ]:
reason_rules = [
    ("Invalid / out-of-period date", bad_date | out_period),
    ("Quality-control sample", df["SampleTypeGroup"].eq("Quality Control").to_numpy()),
    ("Rejected result status", df["ResultStatus"].eq("Rejected").to_numpy()),
    ("Provider-flagged suspect (SUS)", is_suspect),
    ("No numeric value (missing / censored / text)", df["ValueNumeric"].isna().to_numpy()),
    ("Different measure (DO % saturation)", df["UnitStatus"].eq("Different measure").to_numpy()),
    ("Unconvertible / invalid unit", df["UnitStatus"].eq("Unconvertible / invalid unit").to_numpy()),
    ("Physically impossible value", df["IsImpossible"].to_numpy()),
    ("Logical duplicate", df["IsLogicalDuplicate"].to_numpy()),
]
df["ExclusionReason"] = pd.Categorical(
    np.select([m for _, m in reason_rules], [r for r, _ in reason_rules], default="Included"),
    categories=["Included"] + [r for r, _ in reason_rules])
df["IsAnalysisReady"] = df["ExclusionReason"].eq("Included")

reason_counts = df["ExclusionReason"].value_counts().reindex(df["ExclusionReason"].cat.categories)
reason_table = pd.DataFrame({"Rows": reason_counts, "Pct": (100 * reason_counts / len(df)).round(2)})
save_table(reason_table, "01_exclusion_reasons")
reason_table

#### Cell explanation
**What it does:** Bar chart of the number of rows excluded for each reason. It is drawn in the "critical" red because these rows are problems.

**Functions:** `ax.barh` draws horizontal bars, `ax.text` adds the count labels, and `ax.set_xlim` leaves room for the labels.

In [ ]:
r = reason_table.drop("Included").sort_values("Rows")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(r.index, r["Rows"], color=CRITICAL, height=0.6)
for y, v in enumerate(r["Rows"]):
    ax.text(v, y, f"  {v:,}", va="center", fontsize=8, color="#52514e")
ax.set_xlabel("rows (first matching reason)")
ax.set_title(f"Rows excluded from analysis, by reason (analysis-ready: {reason_table.loc['Included', 'Pct']:.1f}%)")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, r["Rows"].max() * 1.2)
save_fig("01_exclusion_reasons")
plt.show()

### 5.1 Statistical outliers (kept and flagged, not removed)
After impossible values are gone, **extreme statistical outliers** are flagged per parameter with Tukey's *far-out* fences (Q1 − 3·IQR, Q3 + 3·IQR). They stay **in** the analysis: high turbidity after a storm or high conductivity in a coastal site is a genuine environmental signal, and exactly what a high-risk dashboard should surface.

#### Cell explanation
**What it does:** Flags **statistical outliers** per parameter with Tukey's "far-out" fences: below Q1 − 3×IQR or above Q3 + 3×IQR (IQR = Q3 − Q1, the middle 50% spread). These rows are **kept**, because an extreme turbidity after a storm is a real event, and it is exactly what a high-risk dashboard must show.

**Functions:** `Series.quantile([0.25, 0.75])` gives Q1 and Q3. `.mean()` of a True/False series gives the share that is True.

**Output to check:** the outlier % per parameter. Skewed parameters (turbidity, conductivity, nitrate) have the most.

In [ ]:
ready = df["IsAnalysisReady"].to_numpy()
df["IsExtremeOutlier"] = False
outlier_rows = []
for p in PARAM_ORDER:
    m = ready & (df["Characteristic"] == p).to_numpy()
    v = df.loc[m, "ValueStd"]
    if v.empty:
        continue
    q1, q3 = v.quantile([0.25, 0.75])
    lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    out = (v < lo) | (v > hi)
    df.loc[v.index[out], "IsExtremeOutlier"] = True
    outlier_rows.append({"Parameter": p, "Q1": q1, "Q3": q3, "LowerFence": lo, "UpperFence": hi,
                         "Outliers": int(out.sum()), "OutlierPct": round(100 * out.mean(), 2)})
outlier_table = pd.DataFrame(outlier_rows).set_index("Parameter")
save_table(outlier_table, "01_extreme_outliers_iqr3")
log_dq("Extreme outlier", "ValueStd beyond Q1-3IQR / Q3+3IQR", df["IsExtremeOutlier"].sum(),
       "Retain + flag IsExtremeOutlier", "Physically possible extremes are real events (storms, hypoxia) and must stay visible for risk analysis.")
outlier_table

## 6. Derived attributes
None of these are original WQP fields. All are **derived**, and each rule is written out in the dictionary in Section 8.

**Thresholds for `ResultBand`, `ExceedanceFlag`, `HighRiskFlag`** (canonical units):

| Parameter | ExceedanceFlag = 1 when | HighRiskFlag = 1 when | Source |
|---|---|---|---|
| pH | < 6.5 or > 9.0 | < 6.0 or > 9.5 | EPA National Recommended Water Quality Criteria, aquatic life (6.5–9.0). High-risk band is project-defined |
| Dissolved oxygen | < 5 mg/L | < 3 mg/L | EPA 1986 *Quality Criteria for Water* (warm-water minimum about 5 mg/L). < 3 is project-defined severe hypoxic stress |
| Nitrate | > 10 mg/L N | > 20 mg/L N | EPA drinking-water MCL = 10 mg/L as N. 2×MCL is project-defined |
| Temperature | > 28 °C | > 32 °C | project-defined, typical upper limits in state warm-water standards |
| Turbidity | > 25 NTU | > 100 NTU | project-defined, common state limit (around 25 NTU above background) |
| Conductivity | > 1,500 µS/cm | > 3,000 µS/cm | project-defined freshwater guidance. **Not applied to `Ocean Water` media** (naturally about 50,000) |

Flags exist **only for analysis-ready rows** (otherwise `<NA>`). Project-defined thresholds are labelled as such in the report.

#### Cell explanation
**What it does:** Creates the **derived calendar and parameter attributes** the assignment lists:
`Year`, `Quarter`, `Month`, `YearMonth`, `WeekStart` (the Monday of the week), `DayOfWeek`, `Season` (Dec–Feb winter, Mar–May spring, Jun–Aug summer, Sep–Nov fall), `TimeBand` (night/morning/afternoon/evening/unknown), `ParameterGroup` (Physical / Chemical / Oxygen / Nutrient), a short parameter name, and a `LocationType` placeholder for Phase 2.

**Functions:** `.dt.year`, `.dt.quarter`, `.dt.month`, `.dt.dayofweek`, `.dt.day_name()` extract date parts. `.dt.to_period("M")` gives the year-month. `.map(dict)` replaces each value using a dictionary. `pd.Categorical(..., ordered=True)` keeps a natural order (Winter < Spring < …) in tables and charts.

In [ ]:
# ---- time attributes ----
d = df["StartDate"]
df["Year"] = d.dt.year.astype("Int16")
df["Quarter"] = d.dt.quarter.astype("Int8")
df["Month"] = d.dt.month.astype("Int8")
df["YearMonth"] = d.dt.to_period("M").astype(str).astype("category")
df["WeekStart"] = (d - pd.to_timedelta(d.dt.dayofweek, unit="D"))      # Monday of the ISO week
df["DayOfWeek"] = pd.Categorical(d.dt.day_name(), categories=[
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], ordered=True)
SEASON_OF_MONTH = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}   # meteorological, N. hemisphere
df["Season"] = pd.Categorical(d.dt.month.map(SEASON_OF_MONTH),
                              categories=["Winter", "Spring", "Summer", "Fall"], ordered=True)
h = df["Hour"].astype("float64").to_numpy()        # NaN where time unknown
df["TimeBand"] = pd.Categorical(
    np.select([np.isnan(h), h < 6, h < 12, h < 18],
              ["Unknown", "Night (00-05)", "Morning (06-11)", "Afternoon (12-17)"], default="Evening (18-23)"),
    categories=["Night (00-05)", "Morning (06-11)", "Afternoon (12-17)", "Evening (18-23)", "Unknown"], ordered=True)

# ---- parameter attributes ----
PARAMETER_GROUP = {"Temperature, water": "Physical", "Turbidity": "Physical", "Conductivity": "Physical",
                   "pH": "Chemical", "Dissolved oxygen (DO)": "Oxygen", "Nitrate": "Nutrient"}
df["ParameterGroup"] = df["Characteristic"].map(PARAMETER_GROUP).astype("category")
df["ParameterShort"] = df["Characteristic"].map(PARAM_SHORT).astype("category")
df["LocationType"] = pd.Categorical(["Not Available (Phase 2 enrichment)"] * len(df))
print(df[["Year", "Quarter", "Month", "Season", "TimeBand", "ParameterGroup", "ParameterShort"]].head())

#### Cell explanation
**What it does:** Creates the outcome attributes, using **documented thresholds** (table above):
* `ResultBand`: an ordered band per parameter, e.g. DO "Critical (<3) / Low (3–5) / Adequate (5–8) / Good (≥8)".
* `ExceedanceFlag` = 1 when the value breaks the guideline (e.g. DO < 5 mg/L, pH outside 6.5–9.0, nitrate > 10 mg/L N).
* `HighRiskFlag` = 1 for the stricter, more serious level (e.g. DO < 3).
* Conductivity flags are **not applied to ocean water**, because seawater is naturally very salty.
* `SiteDailyAverage`: the mean value of the same parameter at the same site on the same day.

Flags exist only for analysis-ready rows; all other rows get `<NA>`.

**Functions:** `outside(v, limits)` (our helper) returns 1 if the value is below the low limit or above the high limit. `pd.array(..., dtype="Int8")` is an integer column that allows missing values. `groupby([...]).transform("mean")` puts the daily mean back on every row.

**Output to check:** the exceedance and high-risk counts and rates per parameter.

In [ ]:
# ResultBand: ordered, parameter-specific bands (labels include their rule)
BAND_RULES = {
    "Temperature, water": [("Cold (<10 °C)", lambda v: v < 10), ("Cool (10-20 °C)", lambda v: v <= 20),
                           ("Warm (20-28 °C)", lambda v: v <= 28), ("Hot (>28 °C)", lambda v: v > 28)],
    "Dissolved oxygen (DO)": [("Critical (<3 mg/L)", lambda v: v < 3), ("Low (3-5 mg/L)", lambda v: v < 5),
                              ("Adequate (5-8 mg/L)", lambda v: v < 8), ("Good (>=8 mg/L)", lambda v: v >= 8)],
    "pH": [("Acidic (<6.5)", lambda v: v < 6.5), ("Normal (6.5-9.0)", lambda v: v <= 9.0), ("Alkaline (>9.0)", lambda v: v > 9.0)],
    "Turbidity": [("Clear (<5 NTU)", lambda v: v < 5), ("Slightly turbid (5-25 NTU)", lambda v: v <= 25),
                  ("Turbid (25-100 NTU)", lambda v: v <= 100), ("Very turbid (>100 NTU)", lambda v: v > 100)],
    "Conductivity": [("Low (<150 uS/cm)", lambda v: v < 150), ("Normal (150-500 uS/cm)", lambda v: v <= 500),
                     ("Elevated (500-1500 uS/cm)", lambda v: v <= 1500), ("High (>1500 uS/cm)", lambda v: v > 1500)],
    "Nitrate": [("Low (<1 mg/L N)", lambda v: v < 1), ("Moderate (1-3 mg/L N)", lambda v: v <= 3),
                ("Elevated (3-10 mg/L N)", lambda v: v <= 10), ("Above MCL (>10 mg/L N)", lambda v: v > 10)],
}
# (low, high): flag = 1 when value < low or value > high; None = no bound on that side
EXCEEDANCE_LIMITS = {"pH": (6.5, 9.0), "Dissolved oxygen (DO)": (5, None), "Nitrate": (None, 10),
                     "Temperature, water": (None, 28), "Turbidity": (None, 25), "Conductivity": (None, 1500)}
HIGH_RISK_LIMITS = {"pH": (6.0, 9.5), "Dissolved oxygen (DO)": (3, None), "Nitrate": (None, 20),
                    "Temperature, water": (None, 32), "Turbidity": (None, 100), "Conductivity": (None, 3000)}


def outside(v, limits):
    lo, hi = limits
    flag = np.zeros(len(v), dtype=bool)
    if lo is not None:
        flag |= v < lo
    if hi is not None:
        flag |= v > hi
    return flag.astype(np.int8)


band = np.full(len(df), None, dtype=object)
exceed = pd.array(np.full(len(df), pd.NA), dtype="Int8")
high_risk = pd.array(np.full(len(df), pd.NA), dtype="Int8")
ocean = df["MediaSubdivision"].eq("Ocean Water").to_numpy()
values = df["ValueStd"].to_numpy()
for p in PARAM_ORDER:
    m = ready & (df["Characteristic"] == p).to_numpy()
    v = values[m]
    band[m] = np.select([rule(v) for _, rule in BAND_RULES[p]], [lbl for lbl, _ in BAND_RULES[p]], default=None)
    if p == "Conductivity":
        m = m & ~ocean                                # flags not meaningful for seawater
        v = values[m]
    exceed[m] = outside(v, EXCEEDANCE_LIMITS[p])
    high_risk[m] = outside(v, HIGH_RISK_LIMITS[p])
df["ResultBand"] = pd.Series(band, index=df.index).astype("category")
df["ExceedanceFlag"] = exceed
df["HighRiskFlag"] = high_risk

# SiteDailyAverage: mean of the same parameter at the same site on the same day (analysis-ready rows only)
df["SiteDailyAverage"] = (df["ValueStd"].where(df["IsAnalysisReady"])
                          .groupby([df["SiteId"], df["StartDate"], df["Characteristic"]], observed=True)
                          .transform("mean"))

flag_summary = (df[df["IsAnalysisReady"]].groupby("ParameterShort", observed=True)
                .agg(Results=("ValueStd", "size"), Exceedances=("ExceedanceFlag", "sum"),
                     HighRisk=("HighRiskFlag", "sum"), ExceedanceRatePct=("ExceedanceFlag", "mean"),
                     HighRiskRatePct=("HighRiskFlag", "mean")))
flag_summary[["ExceedanceRatePct", "HighRiskRatePct"]] *= 100
flag_summary

## 7. Reconciliation: before vs after curation
Row counts and key measures are reconciled so that every raw row is accounted for:
`raw rows = exact duplicates removed + curated rows` and `curated rows = analysis-ready + Σ excluded by reason`.

#### Cell explanation
**What it does:** **Reconciliation**, which proves no row was lost:
* Raw rows = exact duplicates removed + curated rows
* Curated rows = analysis-ready rows + rows excluded for each reason

`assert` stops the notebook if these sums do not match. A second table compares, per parameter, raw row counts against curated and analysis-ready counts, and the raw mean (mixed units) against the standardised mean.

**Functions:** `assert condition` raises an error when the condition is false. `pd.DataFrame({...}).reindex(order)` builds the table in a fixed row order. `display(...)` shows a table in the notebook.

In [ ]:
n_cur = len(df)
n_ready = int(df["IsAnalysisReady"].sum())
flow = pd.DataFrame({"Rows": [N_RAW, n_exact_dup, n_cur, n_cur - n_ready, n_ready]},
                    index=["1. Raw rows in file", "2. Exact duplicates removed", "3. Curated rows (1-2)",
                           "4. Excluded from analysis (flagged, kept)", "5. Analysis-ready rows (3-4)"])
flow["PctOfRaw"] = (100 * flow["Rows"] / N_RAW).round(2)
assert N_RAW == n_exact_dup + n_cur
assert n_cur == n_ready + int(reason_table.drop("Included")["Rows"].sum())
save_table(flow, "01_row_reconciliation")
display(flow)

# per-parameter reconciliation of counts and of the key measure (mean value)
per_param = pd.DataFrame({
    "RawRows": raw_char_counts,
    "CuratedRows": df["Characteristic"].value_counts(),
    "AnalysisReadyRows": df.loc[df["IsAnalysisReady"], "Characteristic"].value_counts(),
    "MeanRawNumeric(any unit)": df.groupby("Characteristic", observed=True)["ValueNumeric"].mean(),
    "MeanStd(analysis-ready)": df[df["IsAnalysisReady"]].groupby("Characteristic", observed=True)["ValueStd"].mean(),
    "CanonicalUnit": pd.Series(CANONICAL_UNIT),
}).reindex(PARAM_ORDER)
per_param["RetainedPct"] = (100 * per_param["AnalysisReadyRows"] / per_param["RawRows"]).round(1)
save_table(per_param, "01_parameter_reconciliation")
per_param

*Why the raw mean and the standardised mean differ:* the raw mean mixes units (°F with °C, mS/cm with µS/cm) and includes impossible values, so it is **not interpretable**. That difference is the reason unit standardisation was needed.

## 8. Data-quality report and derived-attribute dictionary
This is the **Data-Quality Report** the assignment asks for (§4): one row per decision, with the attribute, actual count, treatment and justification.

#### Cell explanation
**What it does:** Turns the decision log `DQ_LOG` into the **Data-Quality Report** required by the assignment. It has one row per decision, with problem, field, actual count, % of raw rows, treatment and justification. It is saved to `reports/tables/01_data_quality_report.csv`.

**Functions:** `pd.DataFrame(list_of_dicts)` makes a table from the log. `pd.option_context("display.max_colwidth", 120)` temporarily shows longer text in the cells.

In [ ]:
dq_report = pd.DataFrame(DQ_LOG)
dq_report["PctOfRawRows"] = (100 * dq_report["Count"] / N_RAW).round(3)
dq_report.index = np.arange(1, len(dq_report) + 1)
save_table(dq_report, "01_data_quality_report")
with pd.option_context("display.max_colwidth", 120):
    display(dq_report)

#### Cell explanation
**What it does:** Writes the **derived-attribute dictionary**: every column we created, the source field(s) it came from, and the exact rule. The assignment requires that derived columns are documented and **not** presented as original source fields.

**Functions:** `pd.DataFrame(list_of_tuples, columns=[...])` builds the table, and `set_index` makes the attribute name the row label.

In [ ]:
DERIVED = [
    ("SampleTypeGroup", "ActivityTypeCode", "QC / Logger-Continuous / Field Measurement / Lab Sample / Unknown (keyword grouping, Section 4.2)"),
    ("IsProvisional", "ResultStatusIdentifier", "Status in Preliminary, Provisional, Raw, Unreviewed"),
    ("ParseStatus, ValueNumeric, CensoredLimit", "ResultMeasureValue", "Numeric parse, '=' stripped, '<x'/'>x' censored (Section 4.3)"),
    ("ValueStd, UnitStd, UnitStatus", "ResultMeasureValue + Unit + Speciation", "Conversion to canonical unit (Section 4.4 table)"),
    ("IsImpossible", "ValueStd", "Outside physical range (Section 4.5)"),
    ("TimeKnown, Hour", "ActivityStartTime", "Valid HH:MM:SS and not a discrete-visit midnight placeholder"),
    ("StartDateTimeLocal / StartDateTimeUTC", "Date + Time + TimeZoneCode", "UTC = local − zone offset (EST −5, CDT −5, PDT −7, ...)"),
    ("CoordinateValid, RegionGroup", "Latitude/Longitude", "Approximate longitudinal bands (Section 4.7). Phase 2 → state-based"),
    ("IsLogicalDuplicate, HasConflictingValue", "Site+datetime+parameter+depth(+value)", "Section 4.8"),
    ("ExclusionReason, IsAnalysisReady", "all of the above", "First matching rule (Section 5)"),
    ("IsExtremeOutlier", "ValueStd", "Beyond Q1 − 3·IQR or Q3 + 3·IQR per parameter"),
    ("Year, Quarter, Month, YearMonth, WeekStart, DayOfWeek", "ActivityStartDate", "Calendar decomposition"),
    ("Season", "Month", "Meteorological, N. hemisphere: DJF Winter, MAM Spring, JJA Summer, SON Fall"),
    ("TimeBand", "Hour", "Night 00–05, Morning 06–11, Afternoon 12–17, Evening 18–23, Unknown"),
    ("ParameterGroup", "CharacteristicName", "Physical (Temp, Turbidity, Conductivity), Chemical (pH), Oxygen (DO), Nutrient (Nitrate)"),
    ("ResultBand", "ValueStd", "Parameter-specific ordered bands (Section 6)"),
    ("ExceedanceFlag", "ValueStd", "Documented regulatory / project thresholds (Section 6)"),
    ("HighRiskFlag", "ValueStd", "Stricter severity thresholds (Section 6)"),
    ("SiteDailyAverage", "ValueStd", "Mean per Site × Date × Characteristic (analysis-ready rows)"),
    ("LocationType", "(not in result file)", "Placeholder 'Not Available' until joined with WQP Station data in Phase 2"),
]
derived_dict = pd.DataFrame(DERIVED, columns=["DerivedAttribute", "SourceField(s)", "Rule"]).set_index("DerivedAttribute")
save_table(derived_dict, "01_derived_attribute_dictionary")
with pd.option_context("display.max_colwidth", 130):
    display(derived_dict)

## 9. Save the curated dataset
The curated table holds **all curated rows** (analysis-ready rows *and* flagged ones). It feeds:
* `02_exploratory_data_analysis.ipynb` (filters `IsAnalysisReady`)
* Phase 2 ETL → staging → dimensional warehouse

On **Kaggle**, the file is written to `/kaggle/working/data/processed/`. *Save Version* the notebook, then attach its output to the EDA notebook (*Add Input → Your Work → this notebook*).

#### Cell explanation
**What it does:** Saves the curated table (all curated rows plus every flag and derived column) as **`wq_curated.parquet`**. This file is the input for notebook 02 and for the Phase 2 warehouse.
* Text columns with few distinct values are stored as categories, so the file is smaller and loads faster.
* `ResultId` stays as text, because every row has a different ID and a category would not help.

**Functions:** `pd.to_numeric` converts depth to a number. `astype("category")` changes the column type. `to_parquet(path, engine="pyarrow")` writes the Parquet file. `Path.stat().st_size` gives the file size.

**On Kaggle:** after this cell finishes, click **Save Version** so the Parquet file is kept as notebook output, then attach it to notebook 02.

In [ ]:
drop_cols = ["ResultRaw"]  # raw text kept until here for traceability. ValueNumeric + ParseStatus replace it
out = df.drop(columns=drop_cols)
out["DepthValue"] = pd.to_numeric(out["DepthValue"], errors="coerce")
for c in out.columns:
    # low-cardinality text -> category (smaller file). ResultId stays text (unique per row)
    if c != "ResultId" and (out[c].dtype == object or pd.api.types.is_string_dtype(out[c].dtype)) \
            and not isinstance(out[c].dtype, pd.CategoricalDtype):
        out[c] = out[c].astype("category")
OUT_PATH = PROCESSED_DIR / ("wq_curated.parquet" if SAMPLE_NROWS is None else "wq_curated_sample.parquet")
out.to_parquet(OUT_PATH, index=False, engine="pyarrow")
print(f"Saved {len(out):,} rows x {out.shape[1]} columns -> {OUT_PATH} ({OUT_PATH.stat().st_size / 1e6:,.0f} MB)")
print("Tables written:", sorted(p.name for p in TABLE_DIR.glob("01_*.csv")))

## 10. Summary
* Every raw row is accounted for (Section 7): exact duplicates are removed, everything else is **kept and flagged** with one explicit `ExclusionReason`.
* Values are standardised to one unit per parameter, which matters for this dataset because about 30% of temperatures are in °F.
* Derived attributes and thresholds are documented (Sections 6 and 8) and are **not** presented as source fields.
* Known gaps carried into Phase 2: `LocationType` and an authoritative state/region need the **WQP Station** file, and `RegionGroup` is a documented approximation until then.

➡ Next: **`02_exploratory_data_analysis.ipynb`**